In [1]:
import polars as pl
from pathlib import Path

In [2]:
FILE_PATH = Path('../../data/customer_churn_records.csv')
df = pl.read_csv(FILE_PATH)

print(df.head())

shape: (5, 18)
┌───────────┬────────────┬──────────┬─────────────┬───┬──────────┬────────────────────┬───────────┬──────────────┐
│ RowNumber ┆ CustomerId ┆ Surname  ┆ CreditScore ┆ … ┆ Complain ┆ Satisfaction Score ┆ Card Type ┆ Point Earned │
│ ---       ┆ ---        ┆ ---      ┆ ---         ┆   ┆ ---      ┆ ---                ┆ ---       ┆ ---          │
│ i64       ┆ i64        ┆ str      ┆ i64         ┆   ┆ i64      ┆ i64                ┆ str       ┆ i64          │
╞═══════════╪════════════╪══════════╪═════════════╪═══╪══════════╪════════════════════╪═══════════╪══════════════╡
│ 1         ┆ 15634602   ┆ Hargrave ┆ 619         ┆ … ┆ 1        ┆ 2                  ┆ DIAMOND   ┆ 464          │
│ 2         ┆ 15647311   ┆ Hill     ┆ 608         ┆ … ┆ 1        ┆ 3                  ┆ DIAMOND   ┆ 456          │
│ 3         ┆ 15619304   ┆ Onio     ┆ 502         ┆ … ┆ 1        ┆ 3                  ┆ DIAMOND   ┆ 377          │
│ 4         ┆ 15701354   ┆ Boni     ┆ 699         ┆ … ┆ 0        

# Group By
O `GROUP BY` funciona da seguinte forma:
1. Agrupa os iguais;
2. Passa uma função resumo, por exemplo:
    - `contagem`, `soma`, `media`, `mediana`, `std`, etc. Nos grupos criados

#### 1) Qual a média de credit_score e balance por geography

In [18]:
df

RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
i64,i64,str,i64,str,str,i64,i64,f64,i64,i64,i64,f64,i64,i64,i64,str,i64
1,15634602,"""Hargrave""",619,"""France""","""Female""",42,2,0.0,1,1,1,101348.88,1,1,2,"""DIAMOND""",464
2,15647311,"""Hill""",608,"""Spain""","""Female""",41,1,83807.86,1,0,1,112542.58,0,1,3,"""DIAMOND""",456
3,15619304,"""Onio""",502,"""France""","""Female""",42,8,159660.8,3,1,0,113931.57,1,1,3,"""DIAMOND""",377
4,15701354,"""Boni""",699,"""France""","""Female""",39,1,0.0,2,0,0,93826.63,0,0,5,"""GOLD""",350
5,15737888,"""Mitchell""",850,"""Spain""","""Female""",43,2,125510.82,1,1,1,79084.1,0,0,5,"""GOLD""",425
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
9996,15606229,"""Obijiaku""",771,"""France""","""Male""",39,5,0.0,2,1,0,96270.64,0,0,1,"""DIAMOND""",300
9997,15569892,"""Johnstone""",516,"""France""","""Male""",35,10,57369.61,1,1,1,101699.77,0,0,5,"""PLATINUM""",771
9998,15584532,"""Liu""",709,"""France""","""Female""",36,7,0.0,1,0,1,42085.58,1,1,3,"""SILVER""",564


In [23]:
print(df.group_by("Geography").agg([
    pl.col("CreditScore").mean(),
    pl.col("Balance").mean()
]))

shape: (3, 3)
┌───────────┬─────────────┬───────────────┐
│ Geography ┆ CreditScore ┆ Balance       │
│ ---       ┆ ---         ┆ ---           │
│ str       ┆ f64         ┆ f64           │
╞═══════════╪═════════════╪═══════════════╡
│ France    ┆ 649.668329  ┆ 62092.636516  │
│ Spain     ┆ 651.333872  ┆ 61818.147763  │
│ Germany   ┆ 651.453567  ┆ 119730.116134 │
└───────────┴─────────────┴───────────────┘


#### 2) Qual a taxa de saída (exited) por gender e geography?

In [22]:
print(df.group_by(['Gender', 'Geography']).agg(
    pl.col('Exited').mean()
))

shape: (6, 3)
┌────────┬───────────┬──────────┐
│ Gender ┆ Geography ┆ Exited   │
│ ---    ┆ ---       ┆ ---      │
│ str    ┆ str       ┆ f64      │
╞════════╪═══════════╪══════════╡
│ Male   ┆ Germany   ┆ 0.278116 │
│ Male   ┆ France    ┆ 0.127497 │
│ Female ┆ France    ┆ 0.20345  │
│ Female ┆ Germany   ┆ 0.375524 │
│ Female ┆ Spain     ┆ 0.212121 │
│ Male   ┆ Spain     ┆ 0.131124 │
└────────┴───────────┴──────────┘


#### 3) Clientes com mais produtos (num_of_products) têm maior saldo médio e maior taxa de churn?

In [25]:
print(df.group_by('NumOfProducts').agg([
    pl.col('Balance').mean(),
    pl.col('Exited').mean()
]))

shape: (4, 3)
┌───────────────┬──────────────┬──────────┐
│ NumOfProducts ┆ Balance      ┆ Exited   │
│ ---           ┆ ---          ┆ ---      │
│ i64           ┆ f64          ┆ f64      │
╞═══════════════╪══════════════╪══════════╡
│ 4             ┆ 93733.135    ┆ 1.0      │
│ 3             ┆ 75458.328195 ┆ 0.827068 │
│ 2             ┆ 51879.145813 ┆ 0.076035 │
│ 1             ┆ 98551.870614 ┆ 0.277144 │
└───────────────┴──────────────┴──────────┘


#### 4) Qual a diferença de salário médio e score de crédito entre clientes ativos e inativos?

In [26]:
print(df.group_by('IsActiveMember').agg([
    pl.col('EstimatedSalary').mean(),
    pl.col('CreditScore').mean()
]))

shape: (2, 3)
┌────────────────┬─────────────────┬─────────────┐
│ IsActiveMember ┆ EstimatedSalary ┆ CreditScore │
│ ---            ┆ ---             ┆ ---         │
│ i64            ┆ f64             ┆ f64         │
╞════════════════╪═════════════════╪═════════════╡
│ 0              ┆ 100767.203854   ┆ 647.973603  │
│ 1              ┆ 99452.965894    ┆ 652.934188  │
└────────────────┴─────────────────┴─────────────┘


#### 5) Para cada card_type, qual o perfil médio dos clientes que reclamaram (complain = 1)?

In [33]:
print(
    df.filter(pl.col('Complain') == 1)
    .group_by('Card Type').agg([
        pl.col('CreditScore').mean(),
        pl.col('Balance').mean(),
        pl.col('Satisfaction Score').mean(),
        pl.len().alias('Count')
    ])
    .sort('Card Type')
)

shape: (4, 5)
┌───────────┬─────────────┬──────────────┬────────────────────┬───────┐
│ Card Type ┆ CreditScore ┆ Balance      ┆ Satisfaction Score ┆ Count │
│ ---       ┆ ---         ┆ ---          ┆ ---                ┆ ---   │
│ str       ┆ f64         ┆ f64          ┆ f64                ┆ u32   │
╞═══════════╪═════════════╪══════════════╪════════════════════╪═══════╡
│ DIAMOND   ┆ 643.001828  ┆ 92092.780091 ┆ 3.014625           ┆ 547   │
│ GOLD      ┆ 647.297521  ┆ 89519.103967 ┆ 3.076446           ┆ 484   │
│ PLATINUM  ┆ 650.307241  ┆ 90265.597808 ┆ 2.972603           ┆ 511   │
│ SILVER    ┆ 641.071713  ┆ 92594.866295 ┆ 2.940239           ┆ 502   │
└───────────┴─────────────┴──────────────┴────────────────────┴───────┘


#### 6) Após agrupar passe uma função personalizada nos grupos

In [20]:
df = pl.DataFrame({
    "regiao": ["norte", "norte", "sul", "sul", "sul"],
    "produto": ["A", "A", "A", "B", "B"],
    "y": [1500, 1200, 800, 200, 2200],
    "y_hat": [110, 1100, 990, 100, 2500],
})

print(df)

shape: (5, 4)
┌────────┬─────────┬──────┬───────┐
│ regiao ┆ produto ┆ y    ┆ y_hat │
│ ---    ┆ ---     ┆ ---  ┆ ---   │
│ str    ┆ str     ┆ i64  ┆ i64   │
╞════════╪═════════╪══════╪═══════╡
│ norte  ┆ A       ┆ 1500 ┆ 110   │
│ norte  ┆ A       ┆ 1200 ┆ 1100  │
│ sul    ┆ A       ┆ 800  ┆ 990   │
│ sul    ┆ B       ┆ 200  ┆ 100   │
│ sul    ┆ B       ┆ 2200 ┆ 2500  │
└────────┴─────────┴──────┴───────┘


In [31]:
# mais de uma coluna
lazzy_rmse = ((pl.col('y') - pl.col('y_hat')) ** 2).mean() ** (1 / 2)
lazzy_mappe = ((pl.col('y') - pl.col('y_hat')).abs() / pl.col('y').abs()).mean().round(2)
(
    df
    .group_by(["regiao", "produto"])
    .agg(
        mae=((pl.col("y") - pl.col("y_hat")).abs().mean()),
        rmse=lazzy_rmse,
        mape=lazzy_mappe,
        qtde=pl.len(),
    )
)

regiao,produto,mae,rmse,mape,qtde
str,str,f64,f64,f64,u32
"""norte""","""A""",745.0,985.418693,0.5,2
"""sul""","""B""",200.0,223.606798,0.32,2
"""sul""","""A""",190.0,190.0,0.24,1


In [35]:
# uma unica coluna
lazzy_expr = (pl.col('y_hat') / 1000).mean()
(
    df
    .group_by(["regiao", "produto"])
    .agg(
        amplitude=(pl.col('y').max() - pl.col('y').min()),
        lazzy_expr=lazzy_expr,
        qtde=pl.len(),
    )
)

regiao,produto,amplitude,lazzy_expr,qtde
str,str,i64,f64,u32
"""sul""","""B""",2000,1.3,2
"""sul""","""A""",0,0.99,1
"""norte""","""A""",300,0.605,2
